In [ ]:
import os
import time
import random
import mlflow
import logging
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
    print("Apple GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU / AMD ROCm GPU
    print("Nvidia/AMD GPU")
else:
    device = torch.device("cpu")
    print("CPU")


random_seed = 42

# Suppress annoying warnings in output
logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
logging.getLogger("mlflow.pytorch").setLevel(logging.ERROR)

# Setup mlflow local server conection
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("z_prediction")
# mlflow.enable_system_metrics_logging()


## Steal Enis model saving logic

In [ ]:
# Base paths
base_model_dir = "../models"
candidates_dir = os.path.join(base_model_dir, "candidates")
champion_dir = os.path.join(base_model_dir, "champion")
metadata_dir = os.path.join(base_model_dir, "metadata")

# Create folders if they don't exist
os.makedirs(candidates_dir, exist_ok=True)
os.makedirs(champion_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)


# =========================
# SAVE CANDIDATE MODEL
# =========================
def save_candidate_model(model, model_name):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


# =========================
# LOAD CHAMPION INFO
# =========================
def load_champion_info():
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None  


# =========================
# SAVE NEW CHAMPION
# =========================
def save_champion_model(model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# =========================
# UPDATE CHAMPION LOGIC
# =========================
def update_champion(model, model_name, mse, mae, hyperparameters):
    current = load_champion_info()

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    elif mae < current["mae"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")

## Use Hugos functions

In [ ]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

## Load in the data

In [ ]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)


x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Smaller dataset to "force" overfitting
"""
x_train = x_train[:50]
y_train = y_train[:50]
"""

## Train the model

Hyperparameters and what we do with them: 

* "epochs"

* "optimizer": (SGD, RMSprop, Adam, …)

* loss functition: MSE, MAE 

* model_type

* Hidden layers: One or several hidden layers with varying  number of nodes, Variants of layer types, Activation and Initialization


* "activation": Video said ReLU was good choice so this is not a function that we are super intrested in.



In [ ]:
def init_weights(m):
        if isinstance(m, nn.Linear):
            #nn.init.xavier_uniform_(m.weight)  # good for Tanh
            nn.init.kaiming_uniform_(m.weight) # good for ReLU
            nn.init.zeros_(m.bias)

    
class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation=nn.ReLU(), dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(activation)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_size = hidden_size
        
        layers.append(nn.Linear(input_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)
    





## Optimize parameters

In [ ]:
param_grid = {
    "hidden_layers": [[512, 256, 128], [256, 128, 64], [512, 256, 128, 64]],
    "learning_rate": [0.001, 0.0005, 0.0001],
    "dropout": [0, 0.2, 0.3],
    "activation": [nn.ReLU(), nn.LeakyReLU()],
    "epochs": [100],
    "patience": 5
}

best_val_loss = float("inf")
best_params = None
run = 0
loss_fn = nn.MSELoss()
verbose = False

for hidden_layers in param_grid["hidden_layers"]:
    for lr in param_grid["learning_rate"]:
        for dropout in param_grid["dropout"]:
            for activation in param_grid["activation"]:
                for epochs in param_grid["epochs"]:

                    run_name = f"hidden_{hidden_layers}_lr_{lr}_dropout_{dropout}_activation_{activation}"

                    parameters = {
                        "layers": hidden_layers,
                        "learning_rate": lr,
                        "dropout": dropout,
                        "activation": activation.__class__.__name__,
                        "epochs": epochs
                        }
                    
                    with mlflow.start_run(run_name=run_name, nested=True):

                        mlflow.log_params(parameters)

                        model = ZPredictor(hidden_layers, activation, dropout).to(device)
                        model.apply(init_weights)
                        optimizer = optim.Adam(model.parameters(), lr=lr)
                        best_val_loss = float("inf")
                        val_loss = float("inf")

                        start_time = time.perf_counter()
                        epochs_no_improve = 0
                        
                        for epoch in range(epochs):

                            # Train step
                            model.train()
                            optimizer.zero_grad()
                            predictions = model(x_train)
                            train_loss = loss_fn(predictions, y_train)
                            train_loss.backward()
                            optimizer.step()

                            # Evaluation step
                            model.eval()
                            with torch.no_grad():
                                val_predictions = model(x_val)
                                val_loss = loss_fn(val_predictions, y_val)
                                val_mae = torch.mean(torch.abs(val_predictions - y_val))
                                val_mse = torch.mean(torch.square(val_predictions - y_val))
                                val_r2 = r2_score(y_val.cpu(), val_predictions.cpu())
                                val_bias = torch.mean(val_predictions - y_val)

                                train_predictions = model(x_train)
                                train_mae = torch.mean(torch.abs(train_predictions - y_train))
                                train_mse = torch.mean(torch.square(train_predictions - y_train))
                                train_r2 = r2_score(y_train.cpu(), train_predictions.cpu())
                                train_bias = torch.mean(train_predictions - y_train)

                            # Log per epoch metrics with step
                            mlflow.log_metrics({
                                "train_loss": train_loss.item(),
                                "train_mae": train_mae.item(),
                                "train_mse": train_mse.item(),
                                "train_r2": train_r2,
                                "train_bias": train_bias.item(),
                                "val_loss": val_loss.item(),
                                "val_mae": val_mae.item(),
                                "val_mse": val_mse.item(),
                                "val_r2": val_r2,
                                "val_bias": val_bias.item()
                            }, step=epoch)

                            if verbose:
                                print(f"Epoch {epoch+1}/{epochs} "
                                    f"train_loss: {train_loss.item():.4f} "
                                    f"train_mae: {train_mae.item():.4f} "
                                    f"train_mse: {train_mse.item():.4f} "
                                    f"val_loss: {val_loss.item():.4f} "
                                    f"val_mae: {val_mae.item():.4f} cm "
                                    f"val_mse: {val_mse.item():.4f}")

                            # Early stopping
                            if val_loss < best_val_loss:
                                best_val_loss = val_loss
                                epochs_no_improve = 0
                                # mlflow.pytorch.log_model(model, name="best_model")
                            else:
                                epochs_no_improve += 1

                            if epochs_no_improve >= param_grid["patience"]:
                                break
                            
                        run += 1
                        
                        save_candidate_model(model, model_name=f"Trial_{run}")
                        update_champion(model, f"Trial_{run}", mse=val_mse, mae=val_mae, hyperparameters=parameters)

                        mlflow.pytorch.log_model(model, name=f"Trial_{run}")